In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv("/Users/anoushka/Desktop/Diversity-Aware-Recommender/data/cleaned_videos.csv")

In [5]:
tfidf = TfidfVectorizer(max_features = 5000, stop_words = 'english')

In [6]:
tfidf_matrix = tfidf.fit_transform(df['content'])

In [7]:
print(tfidf_matrix.shape)

(22427, 5000)


In [8]:
print("Number of videos:", tfidf_matrix.shape[0])
print("Vocabulary size:", tfidf_matrix.shape[1])

Number of videos: 22427
Vocabulary size: 5000


Each video (It's content, consisting of title, description and tags) is treated like a document for TF-IDF. tfidf_matrix contains the vector representation of every video

In [9]:
cosine_sim = cosine_similarity(tfidf_matrix)
print(cosine_sim.shape)

(22427, 22427)


In [10]:
print(np.isclose(cosine_sim[0][0],1.0))

True


In [11]:
# picking a seed video, index : 0
seed_idx = 0
print("Seed video:", df.iloc[seed_idx]['title'])


Seed video: WE WANT TO TALK ABOUT OUR MARRIAGE


In [12]:
# getting similarity scores for this video against all others
sim_scores = list(enumerate(cosine_sim[seed_idx]))

In [13]:
# sorting by similarity score, descending
sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

In [14]:
# get top 10 
top_10 = sim_scores[1:11]

In [15]:
# print recommended titles
for idx, score in top_10:
    print(f"{df.iloc[idx]['title']} -> {round(score, 3)}")

ALL TIME GREATEST AIRPLANE SEAT - Emirates First Class Suite -> 0.954
IS THIS THE CAMERA OF THE FUTURE? -> 0.941
FEEELINGS ABOUT THE IPHONE X -> 0.934
Casey's STUPID NEW LOOK Explained -> 0.922
Mavic AIR DETAILED REVIEW vs. Mavic Pro -> 0.857
$30,000.00 Camera -> 0.473
How to DRAW ON COFFEE STAINS EP. 5!!! -> 0.469
Making new sounds using artificial intelligence -> 0.437
Fishing on SKETCHY Ice ❄️ -> 0.437
Blindtest Oneplus6 మీ అదృష్టాన్ని పరీక్షించుకోండి -> 0.436


Text unrelated to title of seed video is getting a pretty high score, the reason could be videos are uploaded from similar channel. TF-IDF is matching on shared boilerplate text, not actual topic similarity.

In [23]:
print(df.iloc[0]['tags'])
print(df.iloc[0]['description'][:300])

SHANtell martin
SHANTELL'S CHANNEL - https://www.youtube.com/shantellmartin\nCANDICE - https://www.lovebilly.com\n\nfilmed this video in 4k on this -- http://amzn.to/2sTDnRZ\nwith this lens -- http://amzn.to/2rUJOmD\nbig drone - http://tinyurl.com/h4ft3oy\nOTHER GEAR ---  http://amzn.to/2o3GLX5\nSony CAMERA http://


In [24]:
top_video_idx = top_10[0][0]
print(df.iloc[top_video_idx]['tags'])
print(df.iloc[top_video_idx]['description'][:300])

Emirates first class
MY DOPE MERCH - https://shopcaseyneistat.com/\n\nMusic mixed by http://youtube.com/dyalla \nfilmed this video in 4k on this -- http://amzn.to/2sTDnRZ\nwith this lens -- http://amzn.to/2rUJOmD\nbig drone - http://tinyurl.com/h4ft3oy\nOTHER GEAR ---  http://amzn.to/2o3GLX5\nSony CAMERA http://amzn.to/


Noisy boilerplate text(Amazon affiliate links are inflating similarity scores)

In [16]:
import re

In [17]:
def clean_Text(text):
    text = str(text)
    text = re.sub(r'http\S+', '',text) #removing URLS
    text = re.sub(r'\\n', ' ', text)             # removing literal \n characters
    text = re.sub(r'\n', ' ', text)              # removing actual newlines
    text = re.sub(r'[^a-zA-Z\s]', '', text)      # keep only letters and spaces
    text = re.sub(r'\s+', ' ', text)  

    text = text.lower().strip() 
    return text

df['content_clean'] = df['content'].apply(clean_Text)



In [18]:
print("BEFORE:\n", df['content'].iloc[0][:300])
print("\nAFTER:\n", df['content_clean'].iloc[0][:300])

BEFORE:
 WE WANT TO TALK ABOUT OUR MARRIAGE SHANtell martin SHANTELL'S CHANNEL - https://www.youtube.com/shantellmartin\nCANDICE - https://www.lovebilly.com\n\nfilmed this video in 4k on this -- http://amzn.to/2sTDnRZ\nwith this lens -- http://amzn.to/2rUJOmD\nbig drone - http://tinyurl.com/h4ft3oy\nOTHER GE

AFTER:
 we want to talk about our marriage shantell martin shantells channel this video in k on this this lens drone gear camera camera lens sony camera canon camera tripod thing need this for the bendy tripod lens expensive wide lens camera microphone drone cheaper but still great me on intro song by discl


In [19]:
tfidf_matrix = tfidf.fit_transform(df['content_clean'])

In [20]:
print(tfidf_matrix.shape)

(22427, 5000)


In [21]:
# recomputing cosine similarity
cosine_sim = cosine_similarity(tfidf_matrix)

In [22]:
# rerunning the same recommendation for the same seed video
seed_idx = 0
print("Seed video:", df.iloc[seed_idx]['title'])

sim_scores = list(enumerate(cosine_sim[seed_idx]))
sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
top_10 = sim_scores[1:11]

for idx, score in top_10:
    print(f"{df.iloc[idx]['title']} -> {round(score, 3)}")

Seed video: WE WANT TO TALK ABOUT OUR MARRIAGE
IS THIS THE CAMERA OF THE FUTURE? -> 0.796
ALL TIME GREATEST AIRPLANE SEAT - Emirates First Class Suite -> 0.777
FEEELINGS ABOUT THE IPHONE X -> 0.752
Casey's STUPID NEW LOOK Explained -> 0.717
Mavic AIR DETAILED REVIEW vs. Mavic Pro -> 0.616
$30,000.00 Camera -> 0.484
Fishing on SKETCHY Ice ❄️ -> 0.389
Samsung Galaxy S9 Camera: What's New! -> 0.37
Challenge accepted, Casey. -> 0.336
Samsung Galaxy S9 and S9+: Official Introduction -> 0.298


The scores dropped but we are still getting recommendation from the same channel, instead of genuine connection in content. We can conclude that TF-IDF can't distinguish "shares vocabulary because same creator's style" from "shares vocabulary because same topic." A marriage video and a camera review video, if made by the same vlogger with the same filming-and-editing language, looks similar to TF-IDF

In [24]:
df.to_csv('/Users/anoushka/Desktop/Diversity-Aware-Recommender/data/cleaned_videos.csv', index=False)

Now I will implement sentence transformers to impove contextual understanding

In [26]:
from sentence_transformers import SentenceTransformer

In [27]:
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(df['content_clean'].tolist(), show_progress_bar=True)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/701 [00:00<?, ?it/s]

In [28]:
print(embeddings.shape)

(22427, 384)


In [29]:
# same comparison step as TF-IDF, just on embeddings now
embedding_sim = cosine_similarity(embeddings)

print(embedding_sim.shape)  # should be (22427, 22427)

# same seed video, same comparison
seed_idx = 0
print("Seed video:", df.iloc[seed_idx]['title'])

sim_scores = list(enumerate(embedding_sim[seed_idx]))
sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
top_10 = sim_scores[1:11]

for idx, score in top_10:
    print(f"{df.iloc[idx]['title']} -> {round(score, 3)}")

(22427, 22427)
Seed video: WE WANT TO TALK ABOUT OUR MARRIAGE
IS THIS THE CAMERA OF THE FUTURE? -> 0.6190000176429749
PRODUCT PHOTOGRAPHY -> 0.527999997138977
looking back -> 0.5199999809265137
ALL TIME GREATEST AIRPLANE SEAT - Emirates First Class Suite -> 0.5170000195503235
Casey's STUPID NEW LOOK Explained -> 0.5120000243186951
FEEELINGS ABOUT THE IPHONE X -> 0.5049999952316284
TEACHING CASEY HOW TO VLOG -> 0.49399998784065247
World's First Engagement Ring Phone Case -> 0.492000013589859
How I Make Videos -> 0.4909999966621399
technews 177 Redmi Note5 Teaser,Samsung Note 8 Olympic edition,Google Pixel Mic Problems etc -> 0.4909999966621399


Casey's other videos score dropped more and we finally got a marriage related video in the recommendations! ('World's first engagement ring phone case')

In [31]:
seed_idx = 200
print("Seed video:", df.iloc[seed_idx]['title'])

sim_scores = list(enumerate(embedding_sim[seed_idx]))
sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
top_10 = sim_scores[1:11]

for idx, score in top_10:
    print(f"{df.iloc[idx]['title']} -> {round(score, 3)}")

Seed video: Brent Pella - Why You Shouldn't Fly on Spirit Airlines
How Airlines Price Flights -> 0.48399999737739563
Very Cool Conversation Between F 22 Pilots And Boom Operator -> 0.47099998593330383
New $24,000 Singapore Airlines First-Class Suite -> 0.46700000762939453
When Your Uber Driver Takes a Personal Call -> 0.4659999907016754
United Airlines Denies Emotional Support Peacock -> 0.46399998664855957
Leslie Odom Jr. Learned The Secret To Success: Trying -> 0.46299999952316284
Airplane Dad: Part 2 - Cyanide & Happiness Shorts -> 0.4580000042915344
TYPES OF PEOPLE ON AN AEROPLANE | RishhSome -> 0.4580000042915344
FLYING EMIRATES A-380 IN DUBAI MALL 🔥🔥🔥 -> 0.4519999921321869
US Bangla বিমানের পাইলটের শেষ কথা ! দেখুন কি বলেছিলেন পাইলটরা -> 0.45100000500679016


The embedding model clustered an entire "airline/flying culture" topic correctly, across languages again, with no single channel dominating the list this time

Trying to see if giving more importance to titles improves results. Titles often contain the most relevant information of the video, but it could be clickbait as well or been made for optimising number of clicks

In [33]:
# weighted content: title counts 3x more than tags/description
df['content_weighted'] = (df['title'] + ' ') * 3 + df['tags'] + ' ' + df['description']

# clean it the same way as before
df['content_weighted_clean'] = df['content_weighted'].apply(clean_Text)

# encode with the same model
embeddings_weighted = model.encode(df['content_weighted_clean'].tolist(), show_progress_bar=True)

print(embeddings_weighted.shape)

Batches:   0%|          | 0/701 [00:00<?, ?it/s]

(22427, 384)


In [34]:
embedding_sim_weighted = cosine_similarity(embeddings_weighted)

seed_idx = 0
print("Seed video:", df.iloc[seed_idx]['title'])

sim_scores = list(enumerate(embedding_sim_weighted[seed_idx]))
sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
top_10 = sim_scores[1:11]

for idx, score in top_10:
    print(f"{df.iloc[idx]['title']} -> {round(score, 3)}")

Seed video: WE WANT TO TALK ABOUT OUR MARRIAGE
IS THIS THE CAMERA OF THE FUTURE? -> 0.5289999842643738
THE PROPOSAL | Felix & Marzia 💍 -> 0.5130000114440918
PRODUCT PHOTOGRAPHY -> 0.5090000033378601
looking back -> 0.5070000290870667
Honest Husband -> 0.5040000081062317
राजस्थानी बरात फौजी स्टूडियो झुन्झुनू | FOUJI STUDIO JHUNJHUNU | New Marwadi Video 2018 -> 0.5
மணமேடையில் இந்த மணப்பெண்ணுக்கு நடந்த கொடுமையை பாருங்க! | Tamil News | Tamil Seithigal | -> 0.4959999918937683
How I Make Videos -> 0.4950000047683716
TEACHING CASEY HOW TO VLOG -> 0.4880000054836273
വിവാഹ ദിവസം   ദിലീപ് ഭാവനക്ക് കൊടുത്ത സമ്മാനം dileep manju bhavana marriage -> 0.4830000102519989


Giving titles more importance improved results

In [36]:
np.save('/Users/anoushka/Desktop/Diversity-Aware-Recommender/data/embeddings_weighted.npy', embeddings_weighted)
df.to_csv('/Users/anoushka/Desktop/Diversity-Aware-Recommender/data/cleaned_videos.csv', index=False)
print("Saved!")

Saved!
